# Visualize all clusters and members


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplfinance as mpf
import os

# Import your custom modules
from pip_pattern_miner import PIPPatternMiner
from perceptually_important import find_pips

def main():
    path = 'BTCUSDT_1h.csv'
    print("Loading BTC dataset...")
    try:
        data = pd.read_csv(path)
    except FileNotFoundError:
        print(f"Error: Could not find {path}")
        return

    data['date'] = data['date'].astype('datetime64[s]')
    data = data.set_index('date')

    # Un-logged data for candlestick visuals
    plot_data = data.copy()
    plot_data = plot_data[plot_data.index < '01-01-2025']

    # Log-transformed data for the algorithm
    data = np.log(data)
    data = data[data.index < '01-01-2025']
    arr = data['close'].to_numpy()

    print("Training the PIP Pattern Miner...")
    pip_miner = PIPPatternMiner(n_pips=5, lookback=24, hold_period=6)
    pip_miner.train(arr, n_reps=-1)

    num_clusters = len(pip_miner._pip_clusters)
    print(f"\nTraining complete! Found {num_clusters} unique pattern clusters.")

    # --- 1. Create a Master Output Folder ---
    master_folder = "all_clusters_visualized"
    os.makedirs(master_folder, exist_ok=True)
    print(f"Created master folder: '{master_folder}/'")

    plt.style.use('dark_background')

    # --- 2. Loop Through EVERY Cluster ---
    for cluster_id in range(num_clusters):
        cluster_members = pip_miner._pip_clusters[cluster_id]
        num_members = len(cluster_members)
        
        # Create a specific folder for this cluster
        cluster_folder = os.path.join(master_folder, f"cluster_{cluster_id}")
        os.makedirs(cluster_folder, exist_ok=True)
        
        print(f"Processing Cluster {cluster_id} ({num_members} members)...")

        # --- 3. Loop Through EVERY Member in the current Cluster ---
        for i, mem_idx in enumerate(cluster_members):
            # Extract the exact slice of data for this specific pattern instance
            pat_i = pip_miner._unique_pip_indices[mem_idx]
            data_slice = plot_data.iloc[pat_i - pip_miner._lookback + 1: pat_i + 1]
            idx = data_slice.index
            
            # Recalculate PIPs for the visual overlay
            plot_pip_x, plot_pip_y = find_pips(data_slice['close'].to_numpy(), pip_miner._n_pips, 3)
            
            pip_lines = []
            colors = []
            for line_i in range(pip_miner._n_pips - 1):
                l0 = [(idx[plot_pip_x[line_i]], plot_pip_y[line_i]), 
                      (idx[plot_pip_x[line_i + 1]], plot_pip_y[line_i + 1])]
                pip_lines.append(l0)
                colors.append('w')

            # Create the individual plot
            fig, ax = plt.subplots(figsize=(6, 4))
            mpf.plot(data_slice, type='candle', alines=dict(alines=pip_lines, colors=colors), 
                     ax=ax, style='charles', update_width_config=dict(candle_linewidth=1.75))
            
            # Clean up axes for a cleaner look
            ax.set_yticklabels([])
            ax.set_xticklabels([])
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_ylabel("")
            
            # Add a title with the exact date 
            end_date = data_slice.index[-1].strftime('%Y-%m-%d %H:%00')
            ax.set_title(f"Cluster {cluster_id} | Member {i+1}/{num_members}\n(Ending: {end_date})", color='white')

            # Save the file into this cluster's specific folder
            file_path = os.path.join(cluster_folder, f"cluster_{cluster_id}_member_{i:03d}.png")
            plt.savefig(file_path, bbox_inches='tight', dpi=100)
            plt.close(fig) # Critical: Prevents memory leaks
            
        print(f"  -> Saved all {num_members} members for Cluster {cluster_id}.")

    print(f"\nAll done! Check the '{master_folder}/' directory for your files.")

if __name__ == '__main__':
    main()

# Visualize clusters 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Import the PIPPatternMiner class from your original file.
from pip_pattern_miner import PIPPatternMiner

def main():
    print("Loading BTC dataset...")
    path = 'BTCUSDT_1h.csv'
    try:
        data = pd.read_csv(path)
    except FileNotFoundError:
        print(f"Error: Could not find {path}. Please ensure it is in the same directory.")
        return

    # Format the date index
    data['date'] = data['date'].astype('datetime64[s]')
    data = data.set_index('date')

    # Keep a raw copy of the data for mplfinance candlestick plotting
    plot_data = data.copy()
    plot_data = plot_data[plot_data.index < '01-01-2025']

    # Log-transform the data for the actual model training
    data = np.log(data)
    data = data[data.index < '01-01-2025']
    arr = data['close'].to_numpy()

    print("Initializing and training the PIP Pattern Miner...")
    pip_miner = PIPPatternMiner(n_pips=5, lookback=24, hold_period=6)
    pip_miner.train(arr, n_reps=-1)

    num_clusters = len(pip_miner._pip_clusters)
    print(f"\nTraining complete! Found {num_clusters} unique pattern clusters.")

    if num_clusters == 0:
        print("No clusters found to visualize.")
        return

    # --- NEW: Folder Creation Logic ---
    output_folder = "cluster_visualizations"
    os.makedirs(output_folder, exist_ok=True)
    print(f"Created output folder: '{output_folder}/'")

    # Save all clusters (or change to min(5, num_clusters) if you only want a few)
    for i in range(num_clusters):
        print(f"Saving plot for Cluster {i}...")
        
        # --- NEW: Intercept the plot and save it to a file ---
        # We temporarily replace matplotlib's show() function with our own save function
        original_show = plt.show 
        
        def save_and_close(*args, **kwargs):
            file_path = os.path.join(output_folder, f"cluster_{i:02d}.png")
            plt.savefig(file_path, bbox_inches='tight', dpi=150) # Save high-res PNG
            plt.close() # Close the figure to free up memory
            
        plt.show = save_and_close # Apply the override
        
        try:
            # grid_size=3 creates a 3x3 grid (up to 9 examples per cluster)
            pip_miner.plot_cluster_examples(plot_data, cluster_i=i, grid_size=5)
        finally:
            plt.show = original_show # Always restore the original show() function safely

    print(f"\nSuccess! All cluster plots have been saved inside the '{output_folder}' folder.")

if __name__ == '__main__':
    main()

Loading BTC dataset...
Initializing and training the PIP Pattern Miner...
33.84889889456831

Training complete! Found 16 unique pattern clusters.
Created output folder: 'cluster_visualizations/'
Saving plot for Cluster 0...
Saving plot for Cluster 1...
Saving plot for Cluster 2...
Saving plot for Cluster 3...
Saving plot for Cluster 4...
Saving plot for Cluster 5...
Saving plot for Cluster 6...
Saving plot for Cluster 7...
Saving plot for Cluster 8...
Saving plot for Cluster 9...
Saving plot for Cluster 10...
Saving plot for Cluster 11...
Saving plot for Cluster 12...
Saving plot for Cluster 13...
Saving plot for Cluster 14...
Saving plot for Cluster 15...

Success! All cluster plots have been saved inside the 'cluster_visualizations' folder.
